### **Lesson 1: MCP-Model Context Protocol**

MCP (Model Context Protocol) is an open-source standard for connecting AI applications to external systems.
Using MCP, AI applications like Claude or ChatGPT can connect to data sources (e.g. local files, databases), tools (e.g. search engines, calculators) and workflows (e.g. specialized prompts)—enabling them to access key information and perform tasks.

- Build MCP server.
- How to connect with 3rd party server.

**MCP server:** /Users/hemantkp/Desktop/DataScience/GenAI/LANGCHAIN/mcp_server.py

### **Local MCP server**

In [1]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# configure your MCP client to point at your running server
client = MultiServerMCPClient({
    "tool_server": {
        "transport": "stdio",
        "command": "python",
        "args": ["mcp_server.py"],
    }
})

# get tools
tools = await client.get_tools()

# get a prompt by server name and prompt name
prompt_messages = await client.get_prompt("tool_server", "prompt")
prompt_messages = prompt_messages[0].content  # extract the content from the message

# show the prompt content
print(prompt_messages)


You are a helpful assistant about LangChain, LangGraph and LangSmith.

Use:
- web_search for general web queries.
- github_repo_files to read the GitHub README.

Answer questions clearly.



In [2]:
from dotenv import load_dotenv
import os
load_dotenv()

from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.7, #it handles the randomness of the output, higher values (e.g., 0.8) make the output more random, while lower values (e.g., 0.2) make it more focused and deterministic.
    max_tokens=2048, #it limits the maximum number of tokens in the generated response. This helps control the length of the output and can prevent excessively long responses. 
    timeout = 30, #it specifies the maximum time (in seconds) that the model will take to generate a response.If the model takes longer than this time, it will stop and return whatever it has generated so far. This is useful for preventing long waits in case of complex queries or issues with the model.
    max_retries=3, #it defines the maximum number of times to retry a request if it fails due to network issues or other transient errors. This helps improve the robustness of your application by allowing it to recover from temporary problems without crashing.
)

In [3]:
from langchain.agents import create_agent

# Create an agent with the LLM
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt_messages
    )

In [11]:
from pprint import pprint
from langchain.messages import AIMessage, HumanMessage

config = {
    "configurable": {"thread_id": "1"},
}


# prompt asking for a search + summary
response = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(
                content="Use the web_search tool to find LangChain MCP adapter library docs and summarize key points."
            )
        ]
    }
)

pprint(response)

{'messages': [HumanMessage(content='Use the web_search tool to find LangChain MCP adapter library docs and summarize key points.', additional_kwargs={}, response_metadata={}, id='d5dcd3e6-82a7-4369-b789-41a4b6e1d559'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'h8hkzge4g', 'function': {'arguments': '{"query":"LangChain MCP adapter library docs"}', 'name': 'web_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 279, 'total_tokens': 299, 'completion_time': 0.037932274, 'completion_tokens_details': None, 'prompt_time': 0.050267279, 'prompt_tokens_details': None, 'queue_time': 0.061680321, 'total_time': 0.088199553}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019caabc-32cc-78e1-95a4-08f1d881ce3f-0', tool_calls=[{'name': 'web_search', 'args': {'qu

### **Online MCP server**

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
  {
    "time": {
      "transport": "stdio",
      "command": "uvx",
      "args": [
        "mcp-server-time",
        "--local-timezone=America/New_York"
      ]
    }
}
)

tools = await client.get_tools()

In [6]:
from pprint import pprint
from langchain.messages import AIMessage, HumanMessage
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant that provides the current time using the time tool."
)

response = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(
                content="What time is it in New York right now?"
            )
        ]
    }
)               

pprint(response)

{'messages': [HumanMessage(content='What time is it in New York right now?', additional_kwargs={}, response_metadata={}, id='4fc160f8-4c86-4184-b9a0-dd75ca36f9dd'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'qvgk4ndbd', 'function': {'arguments': '{"timezone":"America/New_York"}', 'name': 'get_current_time'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 452, 'total_tokens': 470, 'completion_time': 0.037326573, 'completion_tokens_details': None, 'prompt_time': 0.024371134, 'prompt_tokens_details': None, 'queue_time': 0.048107207, 'total_time': 0.061697707}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019caac3-7265-7642-8a86-22552fb8449f-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'America/New_York'}, 'id': 'qvgk4ndbd', 'type': 

### **Lesson 2: Context and State**

### **Lesson 3: Multi-Agent Systems**

### **Lesson 4: Wedding Planner**